# CliffWalking — Deep Reinforcement Learning on a Mock Machines environment

This notebook trains a **neural-network policy (a DQN)** to solve the classic
[CliffWalking](https://gymnasium.farama.org/environments/toy_text/cliff_walking/)
gridworld — but the environment is **not** written in Python. Its dynamics (the
4×12 grid, the cliff, the goal, the walls) are defined entirely in a **Mock
Machines scenario** (`examples/scenarios/CliffWalkingRL/CliffWalkingRL.yaml`):

![The CliffWalkingRL scenario: a 4×12 grid with the start bottom-left, a row of cliff cells, and the goal bottom-right. Python sends one targeted move per turn and reads the walker's position back as an Arrow table.](../../docs/cliffwalking.svg)

The Python side only:

1. drives the simulation through the `mockmachines` package (in-process, via a
   CGo shared library — observations cross zero-copy as Apache Arrow), and
2. wraps it in a tiny RL environment, then learns a policy with PyTorch and logs
   training curves to **TensorBoard**.

This is the pattern to copy for your own work: **model the environment as a Mock
Machines scenario, then train against it from Python.**

## 1. Setup

From the root of your clone, add the package and the notebook's dependencies to
your uv project, then open this notebook:

```bash
uv add mock-machines jupyter torch tensorboard
uv run jupyter lab examples/notebooks/cliffwalking_dqn.ipynb
```

In [ ]:
import collections, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter

import mockmachines as mm

print('torch', torch.__version__)

## 2. Load the environment scenario

`mm.load` accepts a YAML string, a `.yaml` path, or a scenario directory. Here we
load the `CliffWalkingRL` scenario shipped with the examples.

A single `Walker` lives on a graph of `GridCell` entities. We read the grid once
to learn each cell's `kind` (`normal` / `cliff` / `goal`) and its id — these drive
the reward and termination logic below.

In [ ]:
import os

# Where to load the CliffWalkingRL environment from. `mm.load` accepts a scenario
# directory or a .yaml path. Point MM_SCENARIO at your own copy, or fall back to the
# copy shipped alongside the examples (examples/scenarios/CliffWalkingRL).
def resolve_scenario():
    env = os.environ.get('MM_SCENARIO')
    if env:
        return env
    candidates = [
        os.path.join('..', 'scenarios', 'CliffWalkingRL'),
        os.path.join('examples', 'scenarios', 'CliffWalkingRL'),
    ]
    return next((c for c in candidates if os.path.exists(c)), candidates[0])

SCENARIO = resolve_scenario()
print('scenario source:', SCENARIO)

_probe = mm.load(SCENARIO)
grid = _probe.observe('GridCell').to_pylist()
KIND = {row['ID']: row['kind'] for row in grid}
print('machines:', _probe.machines)
print('grid cells:', len(KIND))
print('start cell id:', _probe.observe('Walker').to_pylist()[0]['current_cell_id'])
print('cliff cells:', sorted(i for i, k in KIND.items() if k == 'cliff'))
print('goal cell:', [i for i, k in KIND.items() if k == 'goal'])
_probe.close()

## 3. From simulation to MDP

A reinforcement-learning agent needs four things; here is how each maps onto the
Mock Machines run:

| RL concept | Mock Machines |
|---|---|
| **state / observation** | the `Walker`'s `current_cell_id`, read from the Arrow observation. We encode it as a one-hot vector over the 48 grid cells. |
| **action** | a targeted event request — `move_north/south/east/west` on the `Walker`. The scenario's `is_not blocked` conditions make an off-grid move a no-op automatically. |
| **transition** | one `sim.step(...)` advances exactly one turn; the Walker moves only when you request it. |
| **reward / termination** | computed here in Python from the cell `kind`: −1 per step, −100 and teleport-to-start on a cliff (episode continues), terminate on the goal. |

Because the Walker only moves when you request a move, *all* of the RL shaping
lives in this thin wrapper — swap the scenario and the same loop trains on a
different world.

In [ ]:
N_CELLS = 48
ACTIONS = ['move_north', 'move_south', 'move_east', 'move_west']

class CliffWalkingEnv:
    """A minimal RL environment over a Mock Machines CliffWalkingRL simulation."""

    def __init__(self, scenario):
        self.sim = mm.load(scenario)
        self.kind = {row['ID']: row['kind'] for row in self.sim.observe('GridCell').to_pylist()}

    def _cell(self):
        return self.sim.observe('Walker').to_pylist()[0]['current_cell_id']

    def _obs(self):
        v = np.zeros(N_CELLS, dtype=np.float32)
        v[self._cell() - 1] = 1.0  # cell ids are 1-based; sentinel is never entered
        return v

    def reset(self):
        self.sim.reset(sim_count=1)
        return self._obs()

    def step(self, action):
        self.sim.step(target='Walker', event=ACTIONS[action])
        kind = self.kind[self._cell()]
        if kind == 'cliff':
            self.sim.reset(sim_count=1)          # -100 and teleport to start...
            return self._obs(), -100.0, False    # ...but the episode continues
        return self._obs(), -1.0, kind == 'goal'

    def close(self):
        self.sim.close()

### Sanity check: the optimal path

The shortest safe route — up off the cliff row, across, and down onto the goal —
is 13 steps, so its return is the textbook **−13**. Let's confirm the environment
agrees before we train anything.

In [ ]:
env = CliffWalkingEnv(SCENARIO)
env.reset()
optimal = ['move_north'] + ['move_east'] * 11 + ['move_south']
ret, done = 0.0, False
for a in optimal:
    _, r, done = env.step(ACTIONS.index(a))
    ret += r
    if done:
        break
print(f'optimal path: return={ret}, reached_goal={done}')
assert done and ret == -13.0
env.close()

## 4. A DQN agent

A small multilayer perceptron maps the one-hot cell to a Q-value per action. We
use the standard DQN ingredients: an experience-replay buffer, a periodically
synced target network, ε-greedy exploration, and a smooth-L1 (Huber) TD loss.

In [ ]:
class QNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(N_CELLS, 128), nn.ReLU(),
            nn.Linear(128, 128), nn.ReLU(),
            nn.Linear(128, len(ACTIONS)),
        )

    def forward(self, x):
        return self.net(x)

## 5. Train, logging to TensorBoard

Every episode we log the return, ε, and step count; every gradient step we log the
TD loss. Watch `episode_return` climb from a few hundred (random falls) toward the
optimal ≈ −13.

In [ ]:
EPISODES = 250
MAX_STEPS = 100
GAMMA = 0.99
BATCH = 128
LR = 1e-3
EPS_START, EPS_END, EPS_DECAY = 1.0, 0.05, 0.97
TARGET_SYNC = 5

env = CliffWalkingEnv(SCENARIO)
q, target = QNet(), QNet()
target.load_state_dict(q.state_dict())
opt = torch.optim.Adam(q.parameters(), lr=LR)
buffer = collections.deque(maxlen=20000)
writer = SummaryWriter('runs/cliffwalking_dqn')

eps = EPS_START
global_step = 0
recent = collections.deque(maxlen=20)

for ep in range(EPISODES):
    s = env.reset()
    ret = 0.0
    steps = 0
    for _ in range(MAX_STEPS):
        if random.random() < eps:
            a = random.randrange(len(ACTIONS))
        else:
            with torch.no_grad():
                a = int(q(torch.from_numpy(s)).argmax())
        s2, r, done = env.step(a)
        buffer.append((s, a, r, s2, done))
        s = s2
        ret += r
        steps += 1
        global_step += 1

        if len(buffer) >= BATCH:
            batch = random.sample(buffer, BATCH)
            ss = torch.from_numpy(np.array([b[0] for b in batch]))
            aa = torch.tensor([b[1] for b in batch])
            rr = torch.tensor([b[2] for b in batch], dtype=torch.float32)
            s2s = torch.from_numpy(np.array([b[3] for b in batch]))
            dd = torch.tensor([b[4] for b in batch], dtype=torch.float32)
            qv = q(ss).gather(1, aa[:, None]).squeeze(1)
            with torch.no_grad():
                td_target = rr + GAMMA * target(s2s).max(1).values * (1 - dd)
            loss = F.smooth_l1_loss(qv, td_target)
            opt.zero_grad()
            loss.backward()
            opt.step()
            writer.add_scalar('loss', loss.item(), global_step)
        if done:
            break

    eps = max(EPS_END, eps * EPS_DECAY)
    if ep % TARGET_SYNC == 0:
        target.load_state_dict(q.state_dict())
    recent.append(ret)
    writer.add_scalar('episode_return', ret, ep)
    writer.add_scalar('epsilon', eps, ep)
    writer.add_scalar('steps', steps, ep)
    if ep % 25 == 0 or ep == EPISODES - 1:
        print(f'ep {ep:3d}  eps {eps:.2f}  return {ret:7.1f}  avg20 {np.mean(recent):7.1f}')

writer.flush()
print('training complete')

## 6. View the curves in TensorBoard

Run the cell below (in Jupyter) to embed TensorBoard, or from a shell:
`tensorboard --logdir runs`.

In [ ]:
# %load_ext tensorboard
# %tensorboard --logdir runs

## 7. Inspect the learned policy

Read the greedy action in every cell and draw it as an arrow grid (row 0 is the
bottom row, with the start at the left, the cliff across the middle, and the goal
at the right). A trained agent walks one row up, straight across, and down onto
the goal — hugging the safe edge above the cliff.

In [ ]:
ARROWS = {0: '^', 1: 'v', 2: '>', 3: '<'}  # north, south, east, west
ROWS, COLS = 4, 12

def greedy_action(cell_id):
    v = np.zeros(N_CELLS, dtype=np.float32)
    v[cell_id - 1] = 1.0
    with torch.no_grad():
        return int(q(torch.from_numpy(v)).argmax())

# cell_id = row * COLS + col + 1, with row 0 the bottom row.
print('Learned greedy policy (S=start, C=cliff, G=goal):')
for row in range(ROWS - 1, -1, -1):
    line = []
    for col in range(COLS):
        cid = row * COLS + col + 1
        kind = KIND.get(cid, 'normal')
        if kind == 'cliff':
            line.append('C')
        elif kind == 'goal':
            line.append('G')
        elif row == 0 and col == 0:
            line.append('S')
        else:
            line.append(ARROWS[greedy_action(cid)])
    print(' '.join(line))

In [ ]:
# Greedy rollout from the start.
env.reset()
ret, done = 0.0, False
for _ in range(MAX_STEPS):
    s = env._obs()
    with torch.no_grad():
        a = int(q(torch.from_numpy(s)).argmax())
    _, r, done = env.step(a)
    ret += r
    if done:
        break
print(f'greedy rollout: return={ret}, reached_goal={done}')
env.close()

## Where to go next

* **Change the environment, not the code.** Point `mm.load` at a different Mock
  Machines scenario directory and the same training loop applies — only the
  observation/action/reward wrapper changes.
* **Richer observations.** `sim.observe(machine)` returns a full pyarrow Table;
  build whatever feature vector your task needs from its columns.
* **Scale up.** The engine runs in-process with zero-copy Arrow hand-off, so the
  step loop is fast enough for millions of environment steps.